<h1 style="text-align: center; font-weight: bold;">
بِسْمِ ٱللَّٰهِ ٱلرَّحْمَٰنِ ٱلرَّحِيمِ
</h1>

Full Name: Mohammadmahdi Bababeyk

Student ID: 4041419005

# Generative Language Models with Transformers: Building a Character-Level GPT

In this notebook, we'll journey through the fundamentals of modern generative language models by building one from scratch. Generative models represent a revolutionary approach in artificial intelligence, capable of creating new content—whether text, images, or other data—that resembles the patterns they've learned from training data.

Transformers, introduced in the landmark paper "Attention Is All You Need," have become the backbone of state-of-the-art language models like GPT, BERT, and their successors. These models leverage a mechanism called **self-attention** to process sequential data, allowing them to capture long-range dependencies and contextual relationships far more effectively than previous architectures like RNNs and LSTMs.

## Learning Objectives

By the end of this notebook, you should be able to:

- **Explain** the core components of the Transformer architecture and their roles in language modeling
- **Implement** multi-head self-attention with causal masking for autoregressive generation
- **Construct** a complete GPT model with embedding layers, transformer blocks, and output projections
- **Train** a generative model and understand the loss landscape during training
- **Generate** coherent text using temperature and top-k sampling techniques
- **Diagnose** and address common issues like CUDA memory constraints during training


## 0.Import Necessary Libraries

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import requests
import numpy as np
from tqdm import tqdm

## 1.Data Preparation: Shakespeare Text Dataset

### Dataset Source

We're using the Tiny Shakespeare dataset, a collection of Shakespeare's works (plays, sonnets, and other writings) compiled by Andrej Karpathy. This dataset contains approximately 1 million characters of Shakespearean text, making it an ideal benchmark for character-level language modeling.

### Character-Level Tokenization Process

1. Vocabulary Creation

- First, we extract all unique characters from the Shakespeare text

- We sort them to create a consistent character-to-index mapping

- The vocabulary size tells us how many unique characters exist in the dataset (typical range: 50-100 for English text)

2. Mapping Dictionaries

   We create two essential mapping dictionaries:

- `stoi` (string-to-index): Maps each character to a unique integer ID

- `itos` (index-to-string): Maps each integer ID back to its corresponding character

3. Encoding and Decoding Functions

- `encode()`: Converts a string into a list of integer token IDs using our vocabulary

- `decode()`: Converts a list of integer token IDs back into a human-readable string

4. Dataset Tokenization

- We encode the entire Shakespeare text into a PyTorch tensor of integers

- The dataset is split into:

    - Training data (90%): Used to train the model's parameters

    - Validation data (10%): Used to evaluate model performance and prevent overfitting

In [2]:
# Download Shakespeare text
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url).text

# Build character-level vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Vocabulary size: {vocab_size}")
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# Tokenize full dataset
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

Vocabulary size: 65


### Practice 1: What are the key characteristics of character-Level tokenization?

Character-level tokenization has several unique trade-offs compared to word or subword (BPE) tokenization:

Small Vocabulary: You only need ~50-100 tokens for English, which saves memory in the embedding and output layers.

No Out-of-Vocabulary (OOV) Issues: Every possible word is just a combination of characters, so the model can technically "spell" its way out of any situation.

Sequence Length: The downside is that sequences become very long. A sentence that is 10 words might be 60+ characters, requiring the model to have a much larger context window to understand long-range dependencies.

Loss of Meaning per Token: Individual characters (like 'a' or 't') carry no inherent semantic meaning compared to subwords like "un-" or "happy."

In [3]:
class CharDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        chunk = self.data[idx:idx + self.block_size + 1]
        x = chunk[:-1]
        y = chunk[1:]
        return x, y

# Hyperparameters (you can tune)
block_size = 128   # context length
batch_size = 512

## 2.Implementing Generative Model

A decoder-only Transformer (like GPT) is made up of decoder blocks without cross-attention, meaning it's designed for autoregressive generation without encoder inputs.
Core Components of a Single Decoder Block:
1. Masked Multi-Head Self-Attention

- Purpose: Allows each token to attend only to previous tokens (causal masking).

- How it works:

    - A mask prevents attending to future positions (upper-triangular matrix of -inf).

    - Computes attention scores between tokens in the same sequence.

    Why masked: Ensures prediction for position i depends only on tokens < i.

2. Position-wise Feed-Forward Network (FFN)

- Structure: Two linear layers with a non-linearity (e.g., GELU) in between.

- Operation: Applied independently to each token position.

- Typical: `Linear → GELU → Linear`, often with expansion factor (e.g., hidden_size → 4× hidden_size → hidden_size).

3. Residual Connections & Layer Normalization

- Residual connection: Adds input of a sub-layer to its output:
    `output = input + sublayer(input)`

- LayerNorm: Applied before (Pre-LN) or after (Post-LN) each sub-layer.

    - Pattern:
    `x = x + Attention(LayerNorm(x))`
    `x = x + FFN(LayerNorm(x))`

### Practice 2: Implement `MultiHeadSelfAttention` with mask


In [4]:
class MultiHeadSelfAttention(nn.Module):
    """
    Causal Multi-Head Self-Attention module for autoregressive Transformer models.

    Projects input embeddings into queries, keys, and values, computes scaled
    dot-product attention with causal masking (to prevent look-ahead), and
    combines results across multiple attention heads.

    Args:
        embed_dim (int): Dimension of input embeddings (C).
        num_heads (int): Number of attention heads.
        dropout (float, optional): Dropout probability on attention weights. Default: 0.1.
        block_size (int): Maximum sequence length (used for causal mask). Must be accessible
                          in the scope where this class is defined (e.g., global or passed in).

    Input:
        x (Tensor): Input tensor of shape (B, T, C), where:
            - B = batch size
            - T = sequence length (<= block_size)
            - C = embed_dim

    Output:
        Tensor: Output tensor of shape (B, T, C), same as input.

    Example:
        >>> attn = MultiHeadSelfAttention(embed_dim=512, num_heads=8, block_size=1024)
        >>> x = torch.randn(2, 64, 512)
        >>> out = attn(x)  # (2, 64, 512)
    """
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        # TODO: Initialize linear projections for Q, K, V, and output
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

        # TODO: Create and register causal mask buffer of shape (1, 1, block_size, block_size)
        # Hint: Use torch.tril(torch.ones(...)) and register_buffer
        mask = torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)
        self.register_buffer('causal_mask', mask)

    def forward(self, x):
        # x: (B, T, C)
        B, T, C = x.shape

        # TODO: Project x to Q, K, V; reshape to (B, num_heads, T, head_dim)
        Q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # Scaled dot-product attention
        attn_scores = (Q @ K.transpose(-2, -1)) / (self.head_dim ** 0.5)  # (B, nh, T, T)

        # TODO: Apply causal mask: replace invalid positions with -inf
        # Use self.causal_mask[:, :, :T, :T] for masking
        attn_scores = attn_scores.masked_fill(self.causal_mask[:, :, :T, :T] == 0, float('-inf'))

        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # TODO: Compute output: weighted sum of V, reshape back to (B, T, C)
        out = attn_weights @ V
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)

### Practice 3: Implement `FeedForward` class

In [5]:
class FeedForward(nn.Module):
    """
    Position-wise Feed-Forward Network for Transformer blocks.

    A two-layer MLP applied independently to each token embedding:
      Linear → GELU → Linear → Dropout

    By default, the hidden dimension is 4× the embedding dimension,
    as used in GPT-family models.

    Args:
        embed_dim (int): Input/output embedding dimension (C).
        ff_dim (int, optional): Hidden layer dimension. If None, defaults to 4 * embed_dim.
        dropout (float, optional): Dropout probability after the final linear layer. Default: 0.1.

    Input:
        x (Tensor): Tensor of shape (B, T, C), where:
            - B = batch size
            - T = sequence length
            - C = embed_dim

    Output:
        Tensor: Same shape as input, (B, T, C).

    Example:
        >>> ff = FeedForward(embed_dim=512)
        >>> x = torch.randn(2, 64, 512)
        >>> out = ff(x)  # (2, 64, 512)
    """
    def __init__(self, embed_dim, ff_dim=None, dropout=0.1):
        super().__init__()
        # TODO: Set ff_dim to 4 * embed_dim if not provided
        ff_dim = ff_dim if ff_dim is not None else 4 * embed_dim

        # TODO: Define self.net as an nn.Sequential with:
        #   - Linear(embed_dim, ff_dim)
        #   - GELU activation
        #   - Linear(ff_dim, embed_dim)
        #   - Dropout(dropout)
        self.net = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),
            nn.Linear(ff_dim, embed_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        # TODO: Pass x through the feed-forward network
        return self.net(x)

### Practice 4: Implement `TransformerBlock` class

In [6]:
class TransformerBlock(nn.Module):
    """
    A single Transformer block using pre-layer normalization (pre-LN) architecture.

    Composed of two sub-layers:
      1. Multi-head self-attention with residual connection
      2. Position-wise feed-forward network with residual connection

    Layer normalization is applied *before* each sub-layer (pre-LN),
    which improves training stability for deep models.

    Args:
        embed_dim (int): Embedding dimension (C).
        num_heads (int): Number of attention heads.
        dropout (float, optional): Dropout probability used in attention and FFN. Default: 0.1.

    Input:
        x (Tensor): Input tensor of shape (B, T, C).

    Output:
        Tensor: Output tensor of shape (B, T, C), same as input.

    Note:
        This block relies on externally defined `MultiHeadSelfAttention` and `FeedForward` classes.

    Example:
        >>> block = TransformerBlock(embed_dim=512, num_heads=8)
        >>> x = torch.randn(2, 64, 512)
        >>> out = block(x)  # (2, 64, 512)
    """
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        # TODO: Initialize two LayerNorm layers (dimension = embed_dim)
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)

        # TODO: Instantiate MultiHeadSelfAttention and FeedForward modules
        # Use embed_dim, num_heads, and dropout as needed
        self.attn = MultiHeadSelfAttention(embed_dim, num_heads, dropout)
        self.ff = FeedForward(embed_dim, dropout=dropout)

    def forward(self, x):
        # Pre-LN style residual blocks
        # TODO: Complete the two residual update steps:
        #   1. Attention sub-layer: x = x + attn(ln1(x))
        #   2. FeedForward sub-layer: x = x + ff(ln2(x))
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

### Practice 5: Implement `GPT` class

In [7]:
class GPT(nn.Module):
    """
    Minimal GPT-style autoregressive language model (decoder-only Transformer).

    Implements token + learned positional embeddings, a stack of causal
    Transformer blocks (pre-LN), weight-tying, and sampling-based generation.

    Args:
        vocab_size (int): Size of the vocabulary.
        embed_dim (int, optional): Embedding dimension. Default: 256.
        num_layers (int, optional): Number of Transformer blocks. Default: 4.
        num_heads (int, optional): Number of attention heads per block. Default: 8.
        block_size (int, optional): Maximum sequence length (context window). Default: 128.
        dropout (float, optional): Dropout probability. Default: 0.1.

    Input (forward):
        idx (LongTensor): Input token indices of shape (B, T), where T ≤ block_size.
        targets (LongTensor, optional): Target token indices of same shape for loss computation.

    Output (forward):
        logits (Tensor): Predicted logits of shape (B, T, vocab_size).
        loss (Tensor or None): Scalar cross-entropy loss if targets provided; else None.

    Methods:
        generate(idx, max_new_tokens, temperature=1.0, top_k=None):
            Generates new tokens autoregressively.

    Example:
        >>> model = GPT(vocab_size=50304)  # GPT-2 vocab size
        >>> idx = torch.randint(0, 50304, (2, 64))
        >>> logits, loss = model(idx, idx)  # teacher-forced training
        >>> generated = model.generate(idx[:, :10], max_new_tokens=20)
    """
    def __init__(self, vocab_size, embed_dim=256, num_layers=4, num_heads=8, block_size=128, dropout=0.1):
        super().__init__()
        self.block_size = block_size

        # TODO: Initialize token and positional embeddings
        #   - token_emb: nn.Embedding(vocab_size, embed_dim)
        #   - pos_emb: nn.Embedding(block_size, embed_dim)
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(block_size, embed_dim)

        # TODO: Initialize dropout layer
        self.dropout = nn.Dropout(dropout)

        # TODO: Create a ModuleList of `num_layers` TransformerBlocks
        self.layers = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, dropout) for _ in range(num_layers)
        ])

        # TODO: Final layer norm and language modeling head
        self.ln_f = nn.LayerNorm(embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size)

        # Weight tying: share weights between embedding and lm_head
        self.token_emb.weight = self.lm_head.weight

        # Apply custom initialization
        self.apply(self._init_weights)

    def _init_weights(self, module):
        # TODO: Initialize Linear and Embedding layers:
        #   - Linear: weight ~ N(0, 0.02), bias = 0 (if exists)
        #   - Embedding: weight ~ N(0, 0.02)
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.block_size, f"Input length {T} > block_size {self.block_size}"

        # TODO: Get token and positional embeddings; sum & apply dropout
        tok_emb = self.token_emb(idx)
        pos_emb = self.pos_emb(torch.arange(T, device=idx.device))
        x = self.dropout(tok_emb + pos_emb)

        # TODO: Pass through all Transformer layers
        for layer in self.layers:
            x = layer(x)

        # TODO: Apply final layer norm and project to logits
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            # Flatten and compute cross-entropy (ignore_index=-1 supports padding)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        Take a conditioning sequence of indices (B, T) and generate max_new_tokens new tokens.
        """
        self.eval()
        for _ in range(max_new_tokens):
            # Crop to last block_size tokens
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature  # (B, vocab_size)
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, idx_next], dim=1)
        return idx

## 3.Setting Up Model

The following code cell establishes the training configuration by first detecting and assigning the appropriate computation device, then initializing a GPT transformer model with specified architectural parameters such as embedding dimensions, layer count, attention heads, and context length. It subsequently prepares the data pipeline by creating training and validation datasets with a character-level loader, and organizes them into shuffled and batched data loaders for efficient processing. Finally, it sets up the AdamW optimizer with a defined learning rate to manage the model's weight updates during the forthcoming training process.

In [8]:
# Hyperparameters
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

model = GPT(
    vocab_size=vocab_size,
    embed_dim=256,
    num_layers=4,
    num_heads=8,
    block_size=block_size,
    dropout=0.1
).to(device)

train_dataset = CharDataset(train_data, block_size)
val_dataset = CharDataset(val_data, block_size)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

Using device: cuda


## 4.Training

In [9]:
def train_epoch():
    model.train()
    total_loss = 0.0
    # Create ONE progress bar for the entire epoch
    pbar = tqdm(train_loader, desc=f"Training", leave=True)
    for xb, yb in pbar:
        xb, yb = xb.to(device), yb.to(device)
        logits, loss = model(xb, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        # Update the progress bar description with current loss
        pbar.set_description(f"Training (loss: {loss.item():.4f})")
    return total_loss / len(train_loader)

def validate():
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        # Create ONE progress bar for the entire validation
        pbar = tqdm(val_loader, desc=f"Validation", leave=True)
        for xb, yb in pbar:
            xb, yb = xb.to(device), yb.to(device)
            _, loss = model(xb, yb)
            total_loss += loss.item()

            # Update the progress bar description with current loss
            pbar.set_description(f"Validation (loss: {loss.item():.4f})")
    return total_loss / len(val_loader)

# Train for a few epochs
for epoch in range(2):
    print(f"\nEpoch {epoch+1}/2")
    train_loss = train_epoch()
    val_loss = validate()
    print(f"\nTrain Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    # Sample generation
    context = torch.tensor([encode("ROMEO:")], dtype=torch.long).to(device)
    generated = model.generate(context, max_new_tokens=100, temperature=0.8, top_k=50)
    print("\n--- Sample Generation ---")
    print(decode(generated[0].tolist()))
    print("------------------------\n")


Epoch 1/2


Validation (loss: 1.5611): 100%|██████████| 218/218 [00:25<00:00,  8.53it/s]



Train Loss: 1.6256 | Val Loss: 1.4999

--- Sample Generation ---
ROMEO:
That will I remember so grieve thee let makes his hand.

TRANIO:
A most convenience sweet ship, and
------------------------


Epoch 2/2


Validation (loss: 1.6795): 100%|██████████| 218/218 [00:25<00:00,  8.52it/s]



Train Loss: 1.1342 | Val Loss: 1.6168

--- Sample Generation ---
ROMEO:
Master had so better: I'll be your old war.

JULIET:
O God! I have done.

Nurse:
Peace it, I am pro
------------------------



### Practice 6

Experiment with hyperparameters and adjust the model architecture to improve performance. Also, include separate inference loop code and save your best model. How can we determine if the model is optimal? Please explain.

In [10]:
# Save the model
torch.save(model.state_dict(), 'shakespeare_gpt.pth')

# Inference Loop
def generate_text(prompt, length=500):
    model.eval()
    input_ids = torch.tensor([encode(prompt)], dtype=torch.long).to(device)
    generated_ids = model.generate(input_ids, max_new_tokens=length, temperature=0.8, top_k=40)
    return decode(generated_ids[0].tolist())

print(generate_text("ROMEO: "))

ROMEO: meantime learn to her to my knee
And meaner 'What comfort than she is made great.
And live I have pilot's and warm
And beat the better words of mine heirs,
And hast thou now melt on the root:
Thou dost the hopeless beaming faith doth all dead.

First Murderer:
If this for your resolve and Lord Angelo.

ROMEO:
She hath spirit to be burrved for the jest.

DUKE VINCENTIO:
Be brief, O heart it, Henry, you are not one.
Edward in the east, short of me and time;
I will not remain your sovereignty,
Your


How do we know if our Shakespeare bot is any good?
Loss Convergence: We look for the "elbow" where the training loss flattens. If the training loss keeps dropping but the validation loss starts rising, we have overfit.

Perplexity: This is the exponent of the cross-entropy loss ($e^{loss}$). It represents how "confused" the model is. Lower is better.

Qualitative "Vibe Check": At early epochs, the model generates gibberish. Mid-training, it learns words but lacks grammar. Optimal training results in "Shakespeare-esque" structures (verse, speaker names in caps) even if the sentences aren't perfectly logical.

Hyperparameter Tuning: For a model this small, increasing embed_dim to 512 or adding 2 more layers usually helps, provided you have the GPU memory (VRAM).

In [12]:
# Initialize model with expert params
model = GPT(
    vocab_size=vocab_size,
    embed_dim=384,
    num_layers=6,
    num_heads=6,
    block_size=256,
    dropout=0.2
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

In [13]:
# Configuration for Early Stopping
patience = 2  # How many epochs to wait for improvement before stopping
counter = 0
best_val_loss = float('inf')

# The main training loop from section 4
for epoch in range(10):  # We can set a higher max, as Early Stopping will protect us
    print(f"\nEpoch {epoch+1}")
    
    # Run training and validation steps defined in the notebook
    train_loss = train_epoch() 
    val_loss = validate()
    
    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    # Check for improvement
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0  # Reset counter
        # Save the best model state
        torch.save(model.state_dict(), 'best_model.pth')
        print("Validation loss improved. Model saved!")
    else:
        counter += 1
        print(f"No improvement in validation loss for {counter} epoch(s).")

    # Exit loop if patience is exceeded
    if counter >= patience:
        print(f"Early stopping triggered! Training stopped at epoch {epoch+1}.")
        break

    # Sample generation to see progress
    context = torch.tensor([encode("ROMEO:")], dtype=torch.long).to(device)
    generated = model.generate(context, max_new_tokens=50, temperature=0.8)
    print(f"\nPreview: {decode(generated[0].tolist())}\n")


Epoch 1


Validation (loss: 1.5497): 100%|██████████| 218/218 [01:01<00:00,  3.54it/s]


Train Loss: 1.6041 | Val Loss: 1.5003
Validation loss improved. Model saved!

Preview: ROMEO:
I would they were ballad-fold all comes for the
l


Epoch 2


Validation (loss: 1.8968): 100%|██████████| 218/218 [01:01<00:00,  3.54it/s]


Train Loss: 0.9297 | Val Loss: 1.9495
No improvement in validation loss for 1 epoch(s).

Preview: ROMEO:
I will be satisfied.

MOPSA:
I had as like you, c


Epoch 3


Validation (loss: 2.4898): 100%|██████████| 218/218 [01:01<00:00,  3.53it/s]

Train Loss: 0.5722 | Val Loss: 2.5238
No improvement in validation loss for 2 epoch(s).
Early stopping triggered! Training stopped at epoch 3.


---
**Note:** This notebook is part of a Deep Learning assignment designed and prepared by [Mahdi Golizadeh](mailto:mahdi.golizadeh@gmail.com).